# Train the encoder tagger

This notebook fine-tunes a small encoder to tag each field of a quotation, scores it on the 1,000 validation documents with the same metrics as the teacher, and exports it to ONNX for the browser demo.

**Run it on a free GPU.** In Colab choose *Runtime → Change runtime type → T4 GPU*; on Kaggle turn on *GPU T4*. Then *Run all*. It needs no API key and costs nothing. Training takes a few minutes on a T4.

**What you get:** `tagger-results.json` (every number with its denominator) and `tagger-onnx.zip` (the model the demo loads). Download both from the file panel (they are in the `quote-distill` folder) and attach them in the project thread.

Nothing here reads the frozen test set. The test-set score is a separate, later step.

In [ ]:
import sys

# The branch with this notebook. After the pull requests merge, change it to "main".
REF = "build/tagger-notebook"
BASE = "jhu-clsp/ettin-encoder-32m"  # 32M-parameter ModernBERT encoder, MIT licence, runs in transformers.js
SEED = 0

!git clone -q --depth 1 --branch {REF} https://github.com/Chi944/quote-distill
%cd quote-distill
!pip install -q -e ".[train]"
sys.path.insert(0, "src")  # this kernel started before the install, so point it at the package directly

## 1. Build the training data

The generator is deterministic, so this rebuilds exactly the files described in `docs/student-data.md`. The hashes below must match.

In [ ]:
import hashlib
import json
from pathlib import Path

!python -m quote_distill.student.data --out data/generated/student

def load(name):
    return [json.loads(line) for line in open(f"data/generated/student/{name}.jsonl", encoding="utf-8")]

train_rows, dev_rows = load("train"), load("dev")
hashes = {n: hashlib.sha256(Path(f"data/generated/student/{n}.jsonl").read_bytes()).hexdigest()[:12] for n in ("train", "dev")}
print(len(train_rows), len(dev_rows), hashes)
assert hashes == {"train": "7bde65f87aa2", "dev": "c15dc3f9e84e"}, "data differs from docs/student-data.md"

## 2. Check the tokenizer before training

Token counts set the maximum input length. The **ceiling** is the score a perfect tagger would get through this tokenizer: the gold tags, read back from tokens and decoded. It should be 1000/1000; anything less is lost before training starts.

In [ ]:
import statistics

from quote_distill.student.tagger import MAX_LENGTH, ceiling, features, load_tokenizer

tok = load_tokenizer(BASE)
f = features(dev_rows, tok)
lengths = sorted(f.lengths)
token_stats = {"median": statistics.median(lengths), "p95": lengths[int(0.95 * (len(lengths) - 1))], "max": lengths[-1]}
dev_ceiling = ceiling(dev_rows, tok)
print("dev tokens per document:", token_stats)
print("truncated at", MAX_LENGTH, "tokens:", f.truncated, "| spans lost:", f.lost, "| tokens straddling two spans:", f.clashes)
print("ceiling, documents exact:", dev_ceiling)

## 3. Train

In [ ]:
import torch

from quote_distill.student.tagger import train

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (this will be slow)")
model = train(train_rows, dev_rows, tok, BASE, "tagger", seed=SEED)

## 4. Score on the validation set

The same report as the teacher pilot: per-field exact match, line-item F1, invented values, arithmetic, and whole documents exact. Every rate shows its denominator.

In [ ]:
from quote_distill.student.tagger import evaluate, predict, torch_logits


def summary(report):
    keys = ("exact_documents", "schema_valid", "critical_fields", "invented", "arithmetic_agrees")
    out = {k: str(report[k]) for k in keys}
    out["line_items_f1"] = report["line_items"]["f1"]
    out["fields"] = {k: str(v) for k, v in report["fields"].items()}
    out["line_fields"] = {k: str(v) for k, v in report["line_fields"].items()}
    return out

torch_report = evaluate(dev_rows, predict(torch_logits(model), tok, dev_rows))
print(json.dumps(summary(torch_report), indent=1))

## 5. Export to ONNX and score the exported model

The browser runs the float32 file: on the first real run the int8 file scored 32/1000 exact on dev against 1000/1000 for float32. Latency here is this machine's CPU with one document at a time; the demo measures the browser separately.

In [ ]:
import time

from quote_distill.student.tagger import export_onnx, onnx_logits

sizes = export_onnx(model, tok, "tagger-onnx")
print({k: f"{v / 1e6:.1f} MB" for k, v in sizes.items()})

results = {}
for name in ("model.onnx", "model_quantized.onnx"):
    run = onnx_logits(f"tagger-onnx/onnx/{name}")
    report = evaluate(dev_rows, predict(run, tok, dev_rows))
    times = []
    for r in dev_rows[:200]:
        enc = tok([r["text"]])
        t0 = time.perf_counter()
        run(enc["input_ids"], enc["attention_mask"])
        times.append((time.perf_counter() - t0) * 1000)
    times.sort()
    results[name] = {"report": summary(report), "failing_documents": report["failing_documents"],
                     "cpu_ms_per_document": {"median": round(times[len(times) // 2], 1),
                                             "p95": round(times[int(0.95 * (len(times) - 1))], 1), "n": len(times)}}
    print(name, results[name]["report"]["exact_documents"], results[name]["cpu_ms_per_document"])

## 6. Save the results

In [ ]:
import shutil
import subprocess

import transformers

commit = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
out = {
    "base": BASE, "ref": REF, "seed": SEED, "max_length": MAX_LENGTH,
    "commit": commit,
    "data_sha256_12": hashes, "train_documents": len(train_rows), "dev_documents": len(dev_rows),
    "dev_tokens_per_document": token_stats, "truncated": str(f.truncated), "ceiling": str(dev_ceiling),
    "torch": summary(torch_report), "onnx": results, "file_bytes": sizes,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "versions": {"torch": torch.__version__, "transformers": transformers.__version__},
}
Path("tagger-results.json").write_text(json.dumps(out, indent=1))
shutil.make_archive("tagger-onnx", "zip", "tagger-onnx")
print("Download tagger-results.json and tagger-onnx.zip from the file panel.")